# Day 2 - Topic 4: Classes and Objects (OOP Core)

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- **OOP (Object-Oriented Programming)** organizes code into **classes** (blueprints) and **objects** (instances built from those blueprints)
- A class bundles **data (attributes)** and **behavior (methods)** together
- **Inheritance** lets a new class reuse and extend an existing class
- **Dunder (double-underscore) methods** like `__init__`, `__str__`, `__len__` let your objects plug into Python's built-in syntax
- Why needed?
  - Real systems model real things: a Dataset, a Model, a Pipeline - each with state + behavior
  - Every major DS library is class-based: `pd.DataFrame`, sklearn's `LinearRegression`, PyTorch's `nn.Module`
- Where used?
  - Writing custom sklearn transformers (requires understanding classes + inheritance)
  - Understanding why `model.fit(X, y)` works and where `model.coef_` comes from
  - Lead-level interviews: "write a small class to represent X" is a standard screen

## 2. Real-Life Analogy

- **Class** = a cookie cutter; **objects** = the cookies. One cutter (blueprint), many cookies (instances), each cookie decorated differently (different attribute values)
- **`__init__`** = the factory setup station: every new cookie passes through it to get its initial shape/decorations
- **`self`** = the word "this particular cookie" - it is how a method knows WHICH cookie it is working on
- **Inheritance** = a child inheriting a parent's recipe book, then adding their own recipes or tweaking a few (overriding)
- **Dunder methods** = universal electrical sockets: implement `__len__` and suddenly the standard plug `len()` works on your object

## 3. Explanation

- `class Name:` defines a class; calling `Name(...)` creates an object (instance)
- `__init__(self, ...)` runs automatically on creation - used to set initial attributes
- `self` is the instance itself - Python passes it automatically as the first argument to every instance method
- **Instance attributes** (`self.x`) belong to each object; **class attributes** (defined directly in the class body) are shared by all instances
- **Inheritance**: `class Child(Parent):` - Child gets all Parent methods/attributes for free
- **Overriding**: Child redefines a Parent method to change behavior; `super()` calls the Parent's version
- **Key dunders:** `__init__` (constructor), `__str__` (print-friendly text), `__repr__` (debug text), `__len__` (enables `len()`), `__eq__` (enables `==` comparison)

> **Trick to remember:** self = "this specific object". Dunders = "teach Python's built-in syntax how to work with MY object".

## 4. Syntax

```python
class Animal:                          # class definition (PascalCase name)
    species_count = 0                  # class attribute (shared by all)

    def __init__(self, name, sound):   # constructor
        self.name = name               # instance attributes (unique per object)
        self.sound = sound

    def speak(self):                   # instance method (self is mandatory)
        return f"{self.name} says {self.sound}"

    def __str__(self):                 # dunder: controls str()/print()
        return f"Animal({self.name})"


class Dog(Animal):                     # inheritance: Dog gets everything Animal has
    def __init__(self, name):
        super().__init__(name, "Woof") # call parent constructor

    def speak(self):                   # override + extend
        return super().speak() + " loudly!"


d = Dog("Rex")                         # create an object
print(d.speak())
```

- `class` - keyword to define a class; names use PascalCase by convention
- `super()` - proxy to the parent class, used to call its methods (most often `super().__init__`)

In [ ]:
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound

    def speak(self):
        return f"{self.name} says {self.sound}"

class Dog(Animal):
    def __init__(self, name):
        super().__init__(name, "Woof")

d = Dog("Rex")
print(d.speak())
print(isinstance(d, Animal))   # True - a Dog IS an Animal


## 5. Examples

### Basic Example

In [ ]:
# Basic: a simple class with attributes and a method
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def give_raise(self, amount):
        self.salary += amount

emp = Employee("Asha", 52000)
emp.give_raise(5000)
print(emp.name, emp.salary)


### Intermediate Example

In [ ]:
# Intermediate: class attribute vs instance attribute + dunder methods
class Dataset:
    total_datasets = 0                    # class attribute - shared

    def __init__(self, name, rows):
        self.name = name                  # instance attributes - per object
        self.rows = rows
        Dataset.total_datasets += 1

    def __len__(self):                    # enables len(obj)
        return self.rows

    def __str__(self):                    # enables clean print(obj)
        return f"Dataset '{self.name}' with {self.rows} rows"

    def __eq__(self, other):              # enables obj1 == obj2
        return self.name == other.name and self.rows == other.rows

train = Dataset("train", 8000)
test = Dataset("test", 2000)

print(train)                    # uses __str__
print(len(test))                # uses __len__
print(train == Dataset("train", 8000))   # uses __eq__ -> True
print(Dataset.total_datasets)   # 3 - shared across ALL instances


- `total_datasets` lives on the CLASS - all instances see the same counter (note it counts 3 because `__eq__`'s comparison created a third Dataset)
- Implementing `__len__`/`__str__`/`__eq__` makes your object feel native to Python - this is exactly why `len(df)` works on a DataFrame
- Without `__eq__`, `==` compares memory identity (like `is`), not values

### Real-World Example

In [ ]:
# Real-world: a minimal sklearn-style transformer - the pattern used in real ML pipelines
class StandardScaler:
    """Simplified version of sklearn's StandardScaler."""

    def __init__(self):
        self.mean_ = None          # trailing underscore = learned during fit (sklearn convention)
        self.std_ = None

    def fit(self, values):
        n = len(values)
        self.mean_ = sum(values) / n
        variance = sum((v - self.mean_) ** 2 for v in values) / n
        self.std_ = variance ** 0.5
        return self                # returning self enables method chaining

    def transform(self, values):
        if self.mean_ is None:
            raise ValueError("Must call fit() before transform()")
        return [(v - self.mean_) / self.std_ for v in values]

    def fit_transform(self, values):
        return self.fit(values).transform(values)     # chaining in action

salaries = [40000, 52000, 60000, 75000, 48000]
scaler = StandardScaler()
scaled = scaler.fit_transform(salaries)

print("Mean learned:", scaler.mean_)
print("Scaled:", [round(s, 2) for s in scaled])


- This IS the sklearn API pattern: `fit` learns state from data, `transform` applies it, learned attributes end with `_`
- `return self` in fit enables `scaler.fit(x).transform(x)` chaining - the same reason Pandas allows method chains
- The state check in transform (raising before fit) mirrors sklearn's NotFittedError - defensive design interviewers love to see
- Understanding this class means you can now write custom transformers that plug into real sklearn Pipelines

## 6. Internal Working

- `Dog("Rex")` triggers two dunders: `__new__` allocates the empty object, then `__init__` initializes it (interviews only expect you to know `__init__`, but naming `__new__` scores points)
- Every object stores its instance attributes in a dict: `obj.__dict__`
- Attribute lookup order: instance `__dict__` -> class -> parent classes, following the **MRO (Method Resolution Order)** - inspect with `ClassName.__mro__`
- `d.speak()` is internally `Dog.speak(d)` - THIS is why `self` must be the first parameter of every method

> **Trick to remember:** obj.method() = Class.method(obj). self is not magic - it is just the object passed in explicitly.

In [ ]:
class Dog:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return f"{self.name} says Woof"

d = Dog("Rex")
print(d.__dict__)          # {'name': 'Rex'} - where instance attributes live
print(d.speak())           # normal call
print(Dog.speak(d))        # identical - proves what self really is
print(Dog.__mro__)         # method resolution order


## 7. Time and Space Complexity

- Attribute access/assignment (`obj.x`): O(1) average - dict lookup underneath
- Method calls: O(1) dispatch overhead (plus the method body's own cost)
- Object creation: O(1) plus whatever `__init__` does
- Inheritance lookup through the MRO: effectively O(1) in practice (cached by CPython)
- Space: each instance costs its `__dict__` - relevant when creating millions of objects (which is why NumPy/Pandas store data in arrays, not per-row Python objects)

## 8. Common Mistakes

- Forgetting `self` in the method definition -> TypeError about argument counts
- Forgetting `self.` when setting attributes in `__init__` (writing `name = name` creates a useless local variable)
- Using a MUTABLE class attribute (like a list) and being surprised it is shared by all instances - same trap as mutable default arguments
- Forgetting `super().__init__(...)` in a child class -> parent attributes never get created
- Comparing objects with `==` without defining `__eq__`, then wondering why two identical-looking objects are not equal

In [ ]:
# The shared mutable class attribute trap
class TeamBad:
    members = []                  # class attribute - SHARED!
    def add(self, name):
        self.members.append(name)

a, b = TeamBad(), TeamBad()
a.add("Asha")
print(b.members)                  # ['Asha'] - b sees a's data!

# Fix: make it an instance attribute in __init__
class TeamGood:
    def __init__(self):
        self.members = []         # fresh list per instance
    def add(self, name):
        self.members.append(name)

x, y = TeamGood(), TeamGood()
x.add("Asha")
print(y.members)                  # [] - correctly independent


## 9. Best Practices

- Class names in PascalCase (`StandardScaler`), methods/attributes in snake_case
- Initialize ALL instance attributes inside `__init__` - no surprise attributes appearing later
- Keep mutable state in instance attributes (set in `__init__`), never in class attributes
- Implement `__repr__`/`__str__` on classes you will debug or log - future you will thank you
- Prefer composition ("has a") over deep inheritance chains ("is a") - flat is better than nested
- Follow the sklearn convention in DS code: learned attributes end with `_`, fit returns self

## 10. Interview Questions

**Beginner**
- Q: What is the difference between a class and an object?
  A: A class is the blueprint defining attributes and methods; an object is a concrete instance created from that blueprint with its own attribute values.
- Q: What is self?
  A: The reference to the specific instance a method is being called on - Python passes it automatically as the first argument (obj.method() is really Class.method(obj)).

**Intermediate**
- Q: What is the difference between a class attribute and an instance attribute?
  A: Class attributes are defined on the class and shared by every instance; instance attributes are set on self (usually in __init__) and unique to each object. Instance attributes shadow class attributes with the same name.
- Q: What does super() do?
  A: It returns a proxy to the parent class, letting you call the parent's methods - most commonly super().__init__() so the parent's initialization still runs in a child class.

**Advanced**
- Q: What is the MRO?
  A: Method Resolution Order - the linearized sequence of classes Python searches when looking up an attribute/method, computed by the C3 algorithm. Critical for understanding multiple inheritance; inspect via ClassName.__mro__.
- Q: What is the difference between __str__ and __repr__?
  A: __str__ returns a user-friendly display string (used by print/str); __repr__ returns an unambiguous developer-oriented string, ideally one that could recreate the object (used in the REPL, debuggers, and containers). If __str__ is missing, Python falls back to __repr__.

## 11. Practice Problems

**Easy**
1. Create a BankAccount class with a balance attribute plus deposit and withdraw methods.
2. Create a Rectangle class with width and height, and an area() method.

**Medium**
3. Add __str__, __eq__, and __len__ (returning perimeter, just for practice) to your Rectangle class and demonstrate each working.
4. Create a SavingsAccount class inheriting from BankAccount that adds an add_interest(rate) method, using super().__init__ properly.

**Hard**
5. Build a MinMaxScaler class following the sklearn pattern (fit learns min_ and max_, transform scales values to the 0-1 range, fit_transform chains both, transform raises an error if called before fit). Test it on a list of salaries.

## 12. Revision Summary

- Class = blueprint, object = instance; `__init__` initializes, `self` = this specific object
- obj.method() is Class.method(obj) - self is just the explicit instance argument
- Instance attributes (self.x) are per-object; class attributes are shared - never share mutable state
- `class Child(Parent)` inherits everything; override to change; `super()` to reuse parent logic
- Dunders plug your objects into Python syntax: `__str__` -> print, `__len__` -> len(), `__eq__` -> ==
- Attribute lookup: instance dict -> class -> MRO chain
- The sklearn pattern (fit/transform, trailing-underscore learned attrs, return self) is THE class design pattern for DS interviews

> **Next topic (Day 2 continues):** @property + Static/Class Methods